# Reproducing Kim, Lycke et al. (2026) from dandiset 001868

Paper. Kim R, Lycke R, Zolotavin P, Montes J, Xie C, Luan L (2026). Learning induces
activation-mechanism-dependent neural plasticity in an intracortical microstimulation task.
*Science Advances* 12, eaef0343. [doi:10.1126/sciadv.aef0343](https://doi.org/10.1126/sciadv.aef0343).
The claims file (`claims/001868/10.1126-sciadv.aef0343.yaml`) was written from the version of
record (PMC13544205) with its supplement; this attempt read the same text.

Dandiset. [DANDI:001868](https://dandiarchive.org/dandiset/001868/0.260715.2016), version
0.260715.2016, pinned because the paper's data availability statement cites it
(doi:10.48324/dandi.001868/0.260715.2016), as the claims file does. It is the only published
version; the draft has the same 85 assets and size and was last modified at publication
(2026-07-15), so there is no other version to compare.

What the dandiset holds. 85 NWB files of 12 mice. Six behavioural mice (ICMS92, 93, 98, 100,
101 and 83, the paper's Mice 1 to 6) have 55 sessions with sorted units (spike times, one mean
waveform, cell type, position on the probe), trial and train tables, wheel position, and in most
sessions volumetric and planar per-ROI dF/F with ROI masks. Six control mice have 30 files with
survey imaging (ROI masks and binary activation per channel and current) and, in some, sorted
units. The behavioural files hold no stimulation-site position in the imaging volume, no
activation calls, and no identity of planar ROIs across sessions; no file holds spike waveforms
in and out of stimulation or multi-channel templates.

Random numbers. Every random draw uses its own `numpy` generator with a fixed seed, given in
the cell that draws.

Outside inputs. None.

| Claim | Status |
|---|---|
| fig2a-threshold-decrease | tested |
| fig2a-steepest-decline-early | tested |
| figs1e-response-times-decrease | tested |
| fig2c-count-increase-fixed-current, fig2b-example-counts, figs2ab-density-peak-200um, fig2e-density-200um-increase, fig2e-radius-increase, figs2de-behaviour-increase-comparison, fig2g-stable-at-threshold | blocked: no stimulation-site position in the imaging volume and no activation calls |
| figs2de-control-stable | blocked: no voxel size, plane spacing or channel-to-contact mapping for the control survey |
| fig3b, fig3df, fig3i, fig3j (two), fig3lm, fig3n (two), figs6 (planar tracking) | blocked: no identity of planar ROIs across sessions |
| fig4-unit-counts | tested |
| fig4d-waveform-separation | blocked: no spike waveforms in and out of stimulation |
| figs3b, figs3c, figs3d, fig4g, fig4ij (two) | tested |
| fig5d, fig5e, figs4b, fig5fg, figs4de, fig5h | tested |
| figs5c-wheel-onset-decrease, figs5-restricted-window-consistent | tested |
| figs7-control-no-change | blocked: the control stimulation protocol differs from the one the methods are defined on |
| figs8-tracked-units | blocked: no multi-channel templates or tracked-unit labels |
| fig6c, fig6d, fig6f, figs9a | tested |
| figs9-coupling-window-and-control | blocked for the controls; the 120 ms part is run and reported |

In [ ]:
import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_v] = "1"  # many small fits; one linear-algebra thread each (protocol, Keep it fast)
import pickle
import time
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy import stats

import icms

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()
if os.environ.get("ICMS_DEV_PICKLE"):  # development only: a cached copy of the same loaded data
    sessions_all = pickle.load(open(os.environ["ICMS_DEV_PICKLE"], "rb"))
else:
    sessions_all = icms.load_behaviour_sessions(threads=12)
print(f"loaded {len(sessions_all)} behavioural sessions from dandiset {icms.DANDISET} "
      f"version {icms.VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
assets = icms.list_assets()
kinds = Counter()
for a in assets:
    subj = a["path"].split("/")[0][4:]
    kinds[(subj in icms.BEHAVIOUR, a["path"].split("_")[-1][:-4])] += 1
print("files:", len(assets), "| behavioural mice:", sum(v for (b, _), v in kinds.items() if b),
      "| control mice:", sum(v for (b, _), v in kinds.items() if not b))
for (b, k), v in sorted(kinds.items()):
    print(f"  {'behaviour' if b else 'control'} files of kind {k}: {v}")

tab = pd.DataFrame([{"subject": s["subject"], "mouse": icms.MOUSE[s["subject"]], "sex": s["sex"],
                     "date": s["date"], "day": s["day"], "week": s["week"],
                     "units": len(s["spikes"]), "trials": len(s["trials"]["trial_index"]),
                     "trains_ephys": len(icms.ephys_trains(s)["start_time"]),
                     "catch": int((s["trials"]["current_uA"] == 0).sum()),
                     "catch_hit": int(s["trials"]["is_hit"][s["trials"]["current_uA"] == 0].sum()),
                     "catch_rt": int((~np.isnan(s["trials"]["response_time"][s["trials"]["current_uA"] == 0])).sum()),
                     "currents": ",".join(str(int(c)) for c in np.unique(s["trials"]["current_uA"])),
                     "channels": ",".join(str(c) for c in np.unique(s["trials"]["stim_channel"]) if c),
                     "wheel": "wheel" in s} for s in sessions_all])
print(tab.to_string())
print("\nsessions per mouse and week:")
print(pd.crosstab(tab["mouse"], tab["week"]))
sessions = [s for s in sessions_all if s["week"] <= 4]
print(f"\nsessions in weeks 0 to 4: {len(sessions)} (the paper: 46 sessions); weeks above 4 dropped: "
      f"{len(sessions_all) - len(sessions)}")
print("catch trials:", tab["catch"].sum(), "| catch trials marked hit:", tab["catch_hit"].sum(),
      "| catch trials with a response time:", tab["catch_rt"].sum())
print("units by stored cell_type:", dict(Counter(ct for s in sessions for ct in s["cell_type"])))

In [ ]:
# Trains and trials: units, ranges and the shared clock
rows = []
for s in sessions_all:
    st, T = s["stim"], s["trials"]
    ti = dict(zip(T["trial_index"], range(len(T["trial_index"]))))
    lag = np.array([st["start_time"][i] - T["start_time"][ti[k]] for i, k in enumerate(st["trial_index"])])
    tr = icms.ephys_trains(s)
    rows.append({"session": f'{s["subject"]} {s["date"]}', "trains": len(st["start_time"]),
                 "trains_70_in_ephys": len(tr["start_time"]),
                 "pulse_counts": ",".join(map(str, sorted(set(st["pulse_count"].tolist())))) if len(st["pulse_count"]) < 0 else
                 f'{st["pulse_count"].min()}-{st["pulse_count"].max()}',
                 "freq": ",".join(map(str, sorted(set(st["frequency_hz"].tolist())))),
                 "phase_us": ",".join(map(str, sorted(set(st["pulse_width_us"].tolist())))),
                 "stim_minus_trial_ms": f"{1000 * lag.min():.1f} to {1000 * lag.max():.1f}",
                 "rt_hit_s": f'{np.nanmin(T["response_time"]):.3f} to {np.nanmax(T["response_time"]):.3f}',
                 "ephys_end_s": round(s["ephys_end"], 1), "last_train_s": round(float(st["stop_time"].max()), 1)})
chk = pd.DataFrame(rows)
print(chk.to_string())
long_out = sum(((s["stim"]["pulse_count"] != 70) & (s["stim"]["stop_time"] < s["ephys_end"])).sum() for s in sessions_all)
print("trains that are not 70 pulses but lie inside the electrophysiology recording:", long_out)
print("hit trials with no response time:",
      sum(int(np.isnan(s["trials"]["response_time"][s["trials"]["is_hit"]]).sum()) for s in sessions_all),
      "| miss trials with a response time:",
      sum(int((~np.isnan(s["trials"]["response_time"][~s["trials"]["is_hit"]])).sum()) for s in sessions_all))

In [ ]:
# Shared clock: spikes and wheel against the train onsets, with the onsets shifted as a contrast.
rng = np.random.default_rng(100)
clock = []
for s in sessions:
    tr = icms.ephys_trains(s)
    on = np.sort(tr["start_time"][np.isin(tr["current_uA"], icms.CURRENTS)])
    if len(on) < 10:
        continue
    sh = on + rng.uniform(1.5, 2.5, len(on))  # shifted into the intertrial period
    allspk = np.sort(np.concatenate(s["spikes"]))
    d_true = (icms.counts_in(allspk, on, 0, 0.7) - icms.counts_in(allspk, on, -0.7, 0)).mean()
    d_shift = (icms.counts_in(allspk, sh, 0, 0.7) - icms.counts_in(allspk, sh, -0.7, 0)).mean()
    T = s["trials"]
    hit = T["is_hit"] & (T["current_uA"] > 0)
    miss = ~T["is_hit"] & (T["current_uA"] > 0)
    w_hit = np.nanmedian(np.abs(icms.wheel_displacement(s, T["start_time"][hit], 0, 0.7)[:, -1])) if "wheel" in s else np.nan
    w_miss = np.nanmedian(np.abs(icms.wheel_displacement(s, T["start_time"][miss], 0, 0.7)[:, -1])) if "wheel" in s else np.nan
    w_pre_hit = np.nanmedian(np.abs(icms.wheel_displacement(s, T["start_time"][hit] - 0.7, 0, 0.7)[:, -1])) if "wheel" in s else np.nan
    clock.append({"session": f'{s["subject"]} {s["date"]}', "delta_spikes_true": d_true,
                  "delta_spikes_shifted": d_shift, "wheel_hit_0_700ms": w_hit, "wheel_miss_0_700ms": w_miss,
                  "wheel_hit_before_onset": w_pre_hit})
clock = pd.DataFrame(clock)
print(clock.round(2).to_string())
print("sessions where the train raises the summed spike count more than the shifted onsets:",
      int((clock.delta_spikes_true > clock.delta_spikes_shifted).sum()), "of", len(clock))
print("sessions where hit trials move the wheel more than miss trials in the 700 ms after trial start:",
      int((clock.wheel_hit_0_700ms > clock.wheel_miss_0_700ms).sum()), "of", len(clock))

In [ ]:
# Recordings deposited twice: identical trial tables or spike trains across files
sig_t, sig_u = defaultdict(list), defaultdict(list)
for s in sessions_all:
    sig_t[hash(np.round(s["trials"]["start_time"], 4).tobytes())].append(f'{s["subject"]} {s["date"]}')
    for i, spk in enumerate(s["spikes"]):
        if len(spk) > 20:
            sig_u[hash(np.round(spk[:200], 5).tobytes())].append(f'{s["subject"]} {s["date"]} unit {i}')
print("identical trial tables:", [v for v in sig_t.values() if len(v) > 1])
dup_u = [v for v in sig_u.values() if len(v) > 1]
print("identical spike trains (first 200 spikes):", len(dup_u), dup_u[:10])

In [ ]:
# Cell type: the stored label against the trough-to-peak of the stored mean waveform
# (below 0.45 ms is a narrow-waveform interneuron; Fig. 4H legend, claims common method)
def trough_to_peak_ms(w, fs=30000.0):
    i = int(np.argmin(w))
    j = i + int(np.argmax(w[i:]))
    return (j - i) / fs * 1000.0

ct_rows = []
for s in sessions:
    for i, w in enumerate(s["waveform"]):
        ct_rows.append({"mouse": icms.MOUSE[s["subject"]], "stored": s["cell_type"][i],
                        "t2p_ms": trough_to_peak_ms(w)})
ct = pd.DataFrame(ct_rows)
ct["t2p_class"] = np.where(ct.t2p_ms < 0.45, "interneuron", "pyramidal")
print(pd.crosstab(ct.stored, ct.t2p_class))
print("trough-to-peak range (ms):", ct.t2p_ms.min().round(3), "to", ct.t2p_ms.max().round(3))
print("waveform_mean shape per unit:", sessions[0]["waveform"].shape[1], "samples (3 ms at 30 kHz), one channel")
print("unit_y_um range:", np.nanmin(np.concatenate([s["unit_y"] for s in sessions])).round(1), "to",
      np.nanmax(np.concatenate([s["unit_y"] for s in sessions])).round(1), "um; electrode rel_y range:",
      min(s["el_y"].min() for s in sessions), "to", max(s["el_y"].max() for s in sessions), "um")

## fig2a-threshold-decrease

> Behavioural detection thresholds fall with learning; thresholds in the late phase (weeks 2 to 4)
> are lower than in the early phase (weeks 0 to 1), median 5.5 against 4.2 uA.

Per channel and block, a cumulative Gaussian with free guess and lapse is fitted with psignifit 4
(main) and by bounded maximum likelihood (variant) to the hit proportion at each current, with the block's catch-trial false-alarm rate
as a 0 uA point; the session threshold of a channel is the mean of its block thresholds. The
dataset stores no thresholds, so only refitted ones are shown. Mouse 6 (ICMS83) has no catch
trials in the files; its fits have no 0 uA point. Effect size r = |Z| / sqrt(n1 + n2) (claims,
common method, for thresholds).

In [ ]:
def r_from_p(p, n1, n2):
    return abs(stats.norm.ppf(p / 2)) / np.sqrt(n1 + n2)

from concurrent.futures import ProcessPoolExecutor
t0 = time.time()
items = []
for s in sessions:
    for d in icms.block_data(s):
        items.append(({"subject": s["subject"], "mouse": icms.MOUSE[s["subject"]], "date": s["date"],
                       "week": s["week"], "phase": s["phase"]}, d))
with ProcessPoolExecutor(4) as ex:
    fits = list(ex.map(icms.fit_block, [d for _, d in items], chunksize=4))
thr_rows = [{**meta, **f} for (meta, _), f in zip(items, fits)]
print(f"{len(thr_rows)} block fits in {time.time() - t0:.0f} s")
blk = pd.DataFrame(thr_rows)
print("block fits:", len(blk), "| with a catch point:", int(blk.has_catch.sum()),
      "| psignifit threshold range", blk.ps_threshold.min().round(2), "to", blk.ps_threshold.max().round(2),
      "| MLE mu at a bound (0 or 30 uA):", int(((blk.mu < 0.01) | (blk.mu > 29.99)).sum()),
      "| guess range", blk.guess.min().round(3), "to", blk.guess.max().round(3),
      "| lapse range", blk.lapse.min().round(3), "to", blk.lapse.max().round(3))
print("blocks per channel and session:", blk.groupby(["subject", "date", "channel"]).size().value_counts().to_dict())
thr = blk.groupby(["subject", "mouse", "date", "week", "phase", "channel"], as_index=False).agg(
    ps=("ps_threshold", "mean"), mu=("mu", "mean"), x50=("x50", "mean"), n_blocks=("block", "size"))
print("channel-session thresholds:", len(thr), "| range (psignifit):", thr.ps.min().round(2), "to", thr.ps.max().round(2), "uA")
res_thr = {}
for name in ["ps", "mu", "x50"]:
    e, l = thr.loc[thr.phase == "early", name], thr.loc[thr.phase == "late", name]
    t = icms.mwu(e, l)
    res_thr[name] = t
    print(f"{name}: early median {t['med1']:.2f} uA (n={t['n1']}), late median {t['med2']:.2f} uA (n={t['n2']}), "
          f"two-sided MWU p = {t['p']:.3g}, r = {r_from_p(t['p'], t['n1'], t['n2']):.2f}")
print("\nby mouse (psignifit):")
print(thr.groupby(["mouse", "phase"]).ps.agg(["median", "size"]).unstack().round(2))
# without Mouse 6, whose blocks lack catch trials
t6 = icms.mwu(thr.loc[(thr.phase == "early") & (thr.subject != "ICMS83"), "ps"],
              thr.loc[(thr.phase == "late") & (thr.subject != "ICMS83"), "ps"])
print(f"\nwithout Mouse 6: early {t6['med1']:.2f} (n={t6['n1']}), late {t6['med2']:.2f} (n={t6['n2']}), p = {t6['p']:.3g}")

**Result: reproduced.** With psignifit, the median channel-session threshold is 5.41 uA early (n = 46) and 3.84 uA late (n = 89), both within tolerance, and late is lower (two-sided Mann-Whitney p = 5.43e-06, r = 0.39). The early n is within 43 +- 4; the late n of 89 is one above 80 +- 8. With the bounded maximum likelihood fit the late median is 3.68 uA, below the tolerance, with the same direction (p = 8.51e-07). Without Mouse 6, whose blocks have no catch trials, early 5.21 and late 3.94 uA (p = 7.04e-05).

## fig2a-steepest-decline-early

> The steepest fall in threshold is between weeks 0 and 1, followed by smaller changes and
> relative stability in weeks 2 to 4.

In [ ]:
wk = thr.groupby("week").ps.agg(["median", "size"])
wk["change_from_previous"] = wk["median"].diff()
print(wk.round(3))
d01 = wk.loc[1, "median"] - wk.loc[0, "median"]
later = wk["change_from_previous"].loc[2:].abs()
print(f"fall from week 0 to 1: {-d01:.3f} uA; largest later week-to-week change: {later.max():.3f} uA")
print("MLE mu:", thr.groupby("week").mu.median().round(3).to_dict(), "| MLE x50:", thr.groupby("week").x50.median().round(3).to_dict())
print("by mouse, median psignifit threshold per week:")
print(thr.pivot_table(index="mouse", columns="week", values="ps", aggfunc="median").round(2))

**Result: reproduced, on the criterion's reading of the figure.** The median threshold falls by 1.772 uA from week 0 to week 1; the largest later week-to-week change is 0.853 uA (week 1 to 2), then weeks 2 to 4 stay between 3.785 and 4.324 uA. With the maximum likelihood fit the margin is narrow (1.232 against 1.220 uA).

## figs1e-response-times-decrease

> Response times follow the same time course as thresholds, falling over training.

Response time of hit trials at 4 to 6 uA. The dataset's `response_time` is measured from trial
start, which precedes the train onset by 0 to 85 ms (data check); the time from train onset is
shown as a variant.

In [ ]:
rt_rows = []
for s in sessions:
    T = s["trials"]
    st = s["stim"]
    on = dict(zip(st["trial_index"], st["start_time"]))
    m = T["is_hit"] & np.isin(T["current_uA"], icms.CURRENTS) & ~np.isnan(T["response_time"])
    for k, t0, rt in zip(T["trial_index"][m], T["start_time"][m], T["response_time"][m]):
        rt_rows.append({"subject": s["subject"], "mouse": icms.MOUSE[s["subject"]], "week": s["week"],
                        "phase": s["phase"], "rt_ms": 1000 * rt,
                        "rt_from_train_ms": 1000 * (t0 + rt - on.get(k, np.nan))})
rt = pd.DataFrame(rt_rows)
print("hit trials at 4 to 6 uA:", len(rt), "| range", rt.rt_ms.min().round(1), "to", rt.rt_ms.max().round(1), "ms")
for col in ["rt_ms", "rt_from_train_ms"]:
    per = rt.pivot_table(index="mouse", columns="week", values=col, aggfunc="median")
    late = rt[rt.phase == "late"].groupby("mouse")[col].median()
    med = per.median()
    print(f"\n{col}: median per mouse and week")
    print(per.round(1))
    print("median over mice per week:", med.round(1).to_dict(), "| late phase:", round(late.median(), 1))
    fall = med[0] - late.median()
    print(f"fall week 0 to late: {fall:.1f} ms; fall week 0 to 1: {med[0] - med[1]:.1f} ms "
          f"({100 * (med[0] - med[1]) / fall:.0f}% of the fall)" if fall > 0 else f"no fall: {fall:.1f} ms")
    t = icms.mwu(rt.loc[rt.phase == "early", col], rt.loc[rt.phase == "late", col])
    print(f"variant, MWU early vs late hit trials: early {t['med1']:.1f} (n={t['n1']}), late {t['med2']:.1f} "
          f"(n={t['n2']}), p = {t['p']:.3g}, rank-biserial r = {t['r']:.2f}")

**Result: reproduced.** The median over mice of the per-mouse median hit response time is 274.9 ms in week 0 and 193.9 ms over the late phase; 75.7 of the 81.0 ms fall (93%) is between weeks 0 and 1. Measured from the train onset the values are 265.4 and 186.8 ms (93% again). Over hit trials, early 190.1 ms against late 173.8 ms (p = 7.19e-17).

## figs5c-wheel-onset-decrease

> Wheel-movement onset decreases from early to late training; median onset is 122 ms in the
> fastest animal (Mouse 1), 135 ms over all hit trials and 129 ms in the late phase.

The stored `wheel_position_processed` is already low-passed at 20 Hz forward and backward (its
description). The response threshold is the 25th percentile of absolute displacement at response
time; the dataset's miss trials have no response time (98 of them do, data check), so the hit
trials define it, and the miss trials with a response time are added as a variant. Onset is
measured from the train onset, the first time |displacement| exceeds 50% of the threshold for 50 ms.
Mice 1 to 5, hit trials at 4 to 6 uA. Readings: threshold per animal and pooled.

In [ ]:
wrows = []
for s in sessions:
    if s["subject"] == "ICMS83":
        continue
    T = s["trials"]
    on = dict(zip(s["stim"]["trial_index"], s["stim"]["start_time"]))
    m = np.isin(T["current_uA"], icms.CURRENTS) & ~np.isnan(T["response_time"])
    for k, t0, rt_, h in zip(T["trial_index"][m], T["start_time"][m], T["response_time"][m], T["is_hit"][m]):
        if k not in on:
            continue
        wrows.append({"si": id(s), "s": s, "subject": s["subject"], "mouse": icms.MOUSE[s["subject"]],
                      "week": s["week"], "phase": s["phase"], "hit": bool(h), "onset_t": on[k],
                      "resp_after_onset": t0 + rt_ - on[k]})
W = pd.DataFrame(wrows)
disp = np.vstack([icms.wheel_displacement(r.s, [r.onset_t], 0, 1.0)[0] for r in W.itertuples()])
ridx = np.clip(np.round(W.resp_after_onset.values * 1000).astype(int), 0, disp.shape[1] - 1)
W["abs_at_resp"] = np.abs(disp[np.arange(len(W)), ridx])
print("trials whose wheel trace is missing (outside the stored signal):", int(np.isnan(disp).all(1).sum()))
print("trials:", len(W), "| hit:", int(W.hit.sum()), "| |displacement| at response, range:",
      W.abs_at_resp.min().round(1), "to", W.abs_at_resp.max().round(1), "a.u.")
wheel_res = {}
for reading in ["per animal", "pooled"]:
    for which in ["hit only", "hit and miss with a response time"]:
        base = W[W.hit] if which == "hit only" else W
        if reading == "pooled":
            thr_w = {sub: np.nanpercentile(base.abs_at_resp, 25) for sub in W.subject.unique()}
        else:
            thr_w = base.groupby("subject").abs_at_resp.quantile(0.25).to_dict()
        hits = W[W.hit].copy()
        hd = disp[W.hit.values]
        hits["onset_ms"] = [1000 * icms.movement_onset(hd[i:i + 1], thr_w[sub])[0]
                            for i, sub in enumerate(hits.subject)]
        m1 = hits.loc[hits.subject == "ICMS92", "onset_ms"].median()
        allm = hits.onset_ms.median()
        latem = hits.loc[hits.phase == "late", "onset_ms"].median()
        t = icms.mwu(hits.loc[hits.phase == "early", "onset_ms"], hits.loc[hits.phase == "late", "onset_ms"])
        wheel_res[(reading, which)] = (m1, allm, latem, t)
        print(f"\nthreshold {reading}, from {which}: thresholds {dict((k, round(v, 1)) for k, v in thr_w.items())}")
        print(f"  median onset Mouse 1 {m1:.1f} ms, all hit trials {allm:.1f} ms, late {latem:.1f} ms; "
              f"no onset found in {int(hits.onset_ms.isna().sum())} of {len(hits)}")
        print(f"  early {t['med1']:.1f} ms (n={t['n1']}) vs late {t['med2']:.1f} ms (n={t['n2']}), "
              f"p = {t['p']:.3g}, rank-biserial r = {t['r']:.2f}")
        print("  median per mouse:", hits.groupby("mouse").onset_ms.median().round(1).to_dict())

**Result: reproduced.** With the response threshold per animal, the median onset is 119.0 ms in Mouse 1, 133.0 ms over all hit trials and 128.0 ms in the late phase, all within tolerance, and onset is earlier late than early (144.0 against 128.0 ms, n = 1307 and 2897 against the paper's 1305 and 2922; p = 6.09e-32, r = -0.23). The pooled threshold gives 118.0, 134.0 and 129.0 ms (p = 1.94e-29).

## Electrophysiology: responses

One response is one unit at one stimulation channel and current in one session (claims, common
method). All currents are computed, since fig. S3D needs the threshold current; the other claims
use 4, 5 and 6 uA. `mod50` and `mod100` are the modulated calls with the code's filters and a
z cap of 50 or 100; `mod_methods` uses the Methods' filters alone. Pulse-locking is computed for
every response modulated under `mod100` (a superset of `mod50`).

In [ ]:
t0 = time.time()
ch_days = icms.channel_days(sessions)
resp_rows = icms.response_table(sessions, rng_seed=0, with_trials=True)
for r in resp_rows:
    r["mod50"] = icms.modulated(r, 50, True, ch_days)
    r["mod100"] = icms.modulated(r, 100, True, ch_days)
    r["mod_methods"] = icms.modulated(r, code_filters=False)
    r["mod"] = r["mod100"]
    r["ctype"] = {"pyramidal": "pyramidal", "Pyramidal Cell": "pyramidal", "interneuron": "interneuron",
                  "Narrow Interneuron": "interneuron"}.get(str(r["cell_type"]), "unknown")
icms.add_pulse_locking(resp_rows, sessions, rng_seed=1)
R = pd.DataFrame([{k: v for k, v in r.items() if k not in ("trial_delta", "trial_hit", "starts")} for r in resp_rows])
R["mouse"] = R.subject.map(icms.MOUSE)
print(f"responses: {len(R)} in {time.time() - t0:.0f} s; modulated mod50 {int(R.mod50.sum())}, mod100 "
      f"{int(R.mod100.sum())}, Methods filters only {int(R.mod_methods.sum())}")
print("channels dropped by the 5-day filter:", sorted(k for k, v in ch_days.items() if v < 5))
print("responses by current:", R.current.value_counts().sort_index().to_dict())
print("z range (mod50):", R.loc[R.mod50, "z"].min().round(2), "to", R.loc[R.mod50, "z"].max().round(2),
      "| time to max range (ms):", R.loc[R.mod50, "tmax_ms"].min(), "to", R.loc[R.mod50, "tmax_ms"].max())
print("responses with 5 trials or fewer:", int((R.n_trials <= 5).sum()), "| z >= 50 among Methods-modulated:",
      int((R.mod_methods & (R.z >= 50)).sum()))
print("PL among mod50 at 4-6 uA:", int(R.loc[R.mod50 & R.current.isin(icms.CURRENTS), "pl"].sum()), "of",
      int((R.mod50 & R.current.isin(icms.CURRENTS)).sum()))
print("modulated (mod50, 4-6 uA) by mouse and phase:")
print(pd.crosstab(R.loc[R.mod50 & R.current.isin(icms.CURRENTS), "mouse"], R.loc[R.mod50 & R.current.isin(icms.CURRENTS), "phase"]))

## fig4-unit-counts

> Electrophysiology recorded 973 units in six animals (672 putative pyramidal, 301 interneurons),
> of which 484 were significantly modulated by the stimulation train (311 pyramidal, 173
> interneurons).

Units are counted per session (reading a); the files carry no identity of a unit across
sessions, so reading (b), unique units per animal, cannot be formed. Cell type is the stored
`cell_type` (ICMS83 uses other spellings, which are the same labels), with the trough-to-peak of
the stored mean waveform as a variant.

In [ ]:
def unit_counts(sel_sessions, mod_col, currents):
    rows = []
    for si, s in enumerate(sessions):
        if s not in sel_sessions:
            continue
        sub = R[(R.si == si) & R.current.isin(currents)]
        modu = set(sub.loc[sub[mod_col], "unit"])
        for u in range(len(s["spikes"])):
            ct_ = {"pyramidal": "pyramidal", "Pyramidal Cell": "pyramidal", "interneuron": "interneuron",
                   "Narrow Interneuron": "interneuron"}.get(str(s["cell_type"][u]), "unknown")
            rows.append({"mouse": icms.MOUSE[s["subject"]], "ctype": ct_, "mod": u in modu,
                         "t2p": "interneuron" if trough_to_peak_ms(s["waveform"][u]) < 0.45 else "pyramidal"})
    return pd.DataFrame(rows)

uc_main = None
for label, sel, mod_col, cur in [("weeks 0-4, mod50, 4-6 uA (main)", sessions, "mod50", icms.CURRENTS),
                                 ("weeks 0-4, Methods filters, 4-6 uA", sessions, "mod_methods", icms.CURRENTS),
                                 ("weeks 0-4, mod50, all currents", sessions, "mod50", tuple(sorted(R.current.unique())))]:
    U = unit_counts(sel, mod_col, cur)
    if uc_main is None:
        uc_main = U
    print(f"{label}: units {len(U)}, pyramidal {int((U.ctype == 'pyramidal').sum())}, interneurons "
          f"{int((U.ctype == 'interneuron').sum())}, unknown {int((U.ctype == 'unknown').sum())}; modulated "
          f"{int(U['mod'].sum())} (pyramidal {int((U['mod'] & (U.ctype == 'pyramidal')).sum())}, interneurons "
          f"{int((U['mod'] & (U.ctype == 'interneuron')).sum())})")
U = uc_main
print("by trough-to-peak: all", U.t2p.value_counts().to_dict(), "| modulated", U.loc[U["mod"], "t2p"].value_counts().to_dict())
# variant: all 55 sessions, weeks 5 and 6 included
extra = [s for s in sessions_all if s["week"] > 4]
rx = icms.response_table(extra, rng_seed=5, currents=icms.CURRENTS)
ch_days_all = icms.channel_days(sessions_all)
mod_x = {(id(extra[r["si"]]), r["unit"]) for r in rx if icms.modulated(r, 50, True, ch_days_all)}
ux = pd.DataFrame([{"ctype": {"pyramidal": "pyramidal", "Pyramidal Cell": "pyramidal", "interneuron": "interneuron",
                              "Narrow Interneuron": "interneuron"}.get(str(s["cell_type"][u]), "unknown"),
                    "mod": (id(s), u) in mod_x} for s in extra for u in range(len(s["spikes"]))])
UA = pd.concat([U[["ctype", "mod"]], ux])
print(f"variant, all 55 sessions: units {len(UA)}, pyramidal {int((UA.ctype == 'pyramidal').sum())}, interneurons "
      f"{int((UA.ctype == 'interneuron').sum())}; modulated {int(UA['mod'].sum())} (pyramidal "
      f"{int((UA['mod'] & (UA.ctype == 'pyramidal')).sum())}, interneurons {int((UA['mod'] & (UA.ctype == 'interneuron')).sum())})")
print("per mouse:", U.groupby("mouse").size().to_dict())
t2p_all = np.array([trough_to_peak_ms(w) for s in sessions for w in s["waveform"]])
h, e = np.histogram(t2p_all, bins=np.arange(0, 1.2, 0.05))
print("trough-to-peak histogram (0.05 ms bins from 0):", h.tolist())

**Result: partially reproduced.** In weeks 0 to 4 the files hold 795 unit-sessions (584 pyramidal, 204 interneurons, 7 unknown), of which 488 are modulated (339 pyramidal, 146 interneurons): the modulated total and the modulated pyramidal count are within tolerance, the other four counts are below it (2 of 6). Over all 55 sessions there are 999 units (732 and 260) and 585 modulated (405 and 177), 3 of 6 within tolerance. Reading (b) cannot be formed. The trough-to-peak histogram has a dip in the 0.45 to 0.50 ms bin (22 units) between 59 and 44, so the distribution is bimodal about 0.45 ms.

## figs3b-modulation-increase-all

> The modulation z-score of all modulated responses at fixed currents increases from the early to
> the late phase at 4, 5 and 6 uA.

In [ ]:
def phase_test(df, col, label, alternative="two-sided"):
    out = {}
    for c in icms.CURRENTS:
        d = df[df.current == c]
        t = icms.mwu(d.loc[d.phase == "early", col], d.loc[d.phase == "late", col], alternative)
        out[c] = t
        print(f"{label} {int(c)} uA: early median {t['med1']:.3f} (n={t['n1']}), late median {t['med2']:.3f} "
              f"(n={t['n2']}), p = {t['p']:.3g}, r = {t['r']:.2f}")
    return out

M50 = R[R.mod50 & R.current.isin(icms.CURRENTS) & R.phase.notna()]
res_s3b = phase_test(M50, "z", "z")
print("variant, Methods filters only:")
_ = phase_test(R[R.mod_methods & R.current.isin(icms.CURRENTS) & R.phase.notna()], "z", "z")
print("by mouse (median z, early / late, all three currents):")
print(M50.pivot_table(index="mouse", columns="phase", values="z", aggfunc="median").round(2))

**Result: reproduced.** The modulation z-score of modulated responses is higher late than early at 4 uA (2.706 against 3.765, p = 9.57e-05, r = 0.18), 5 uA (3.399 against 3.875, p = 0.00656, r = 0.12) and 6 uA (3.559 against 4.570, p = 0.000404, r = 0.17). The response counts (191/738, 284/498, 300/256) are close to the paper's (189/729, 283/489, 303/250).

## figs3c-latency-all

> Time to maximum firing rate of all modulated responses decreases at 5 uA but not significantly at
> 4 or 6 uA.

In [ ]:
res_s3c = phase_test(M50, "tmax_ms", "time to max (ms)")

**Result: partially reproduced.** Time to maximum falls at 5 uA (80 to 45 ms, p = 0.00522) and does not change significantly at 6 uA (p = 0.168), as the paper reports, but it also falls significantly at 4 uA (75 to 55 ms, p = 0.0465, r = -0.09), where the paper reports p = 0.18.

## figs3d-modulation-stable-at-threshold

> Modulation at each session's detection threshold does not change over training.

The session threshold of a channel (psignifit, fig2a) rounded to the nearest uA picks the current.

In [ ]:
thr_cur = {(r.subject, r.date, r.channel): int(np.round(r.ps)) for r in thr.itertuples()}
R["thr_cur"] = [thr_cur.get((a, b, c), -1) for a, b, c in zip(R.subject, R.date, R.channel)]
AT = R[R.mod50 & (R.current == R.thr_cur) & R.phase.notna()]
per_cs = AT.groupby(["subject", "date", "channel", "phase"], as_index=False).z.median()
ta = icms.mwu(per_cs.loc[per_cs.phase == "early", "z"], per_cs.loc[per_cs.phase == "late", "z"])
tb = icms.mwu(AT.loc[AT.phase == "early", "z"], AT.loc[AT.phase == "late", "z"])
print("threshold currents used:", AT.current.value_counts().sort_index().to_dict())
print(f"(a) channel-session medians: early {ta['med1']:.2f} (n={ta['n1']}), late {ta['med2']:.2f} (n={ta['n2']}), "
      f"p = {ta['p']:.3g}, r = {ta['r']:.2f}")
print(f"(b) all responses: early {tb['med1']:.2f} (n={tb['n1']}), late {tb['med2']:.2f} (n={tb['n2']}), "
      f"p = {tb['p']:.3g}, r = {tb['r']:.2f}")
# x50 threshold reading
thr_cur2 = {(r.subject, r.date, r.channel): int(np.round(r.x50)) if not np.isnan(r.x50) else -1 for r in thr.itertuples()}
AT2 = R[R.mod50 & (R.current == [thr_cur2.get((a, b, c), -1) for a, b, c in zip(R.subject, R.date, R.channel)]) & R.phase.notna()]
p2 = AT2.groupby(["subject", "date", "channel", "phase"], as_index=False).z.median()
t2 = icms.mwu(p2.loc[p2.phase == "early", "z"], p2.loc[p2.phase == "late", "z"])
print(f"(a) with the MLE x50 threshold: early n={t2['n1']}, late n={t2['n2']}, p = {t2['p']:.3g}, r = {t2['r']:.2f}")

**Result: reproduced.** At the current nearest each session's threshold, the channel-session median z-scores do not differ between phases (early 3.92, n = 42; late 3.46, n = 87; p = 0.241, r = -0.13), nor do all modulated responses (p = 0.612). The n are larger than the paper's 21 and 34.

## fig4g-modulated-count-mouse1

> The number of modulated units rose moderately in some animals (Mouse 1 at 4 uA), with no
> significant change at group level.

In [ ]:
cnt = (R[(R.current == 4) & R.phase.notna()].groupby(["subject", "mouse", "date", "channel", "phase"], as_index=False)
       .mod50.sum().rename(columns={"mod50": "n_mod"}))
m1 = cnt[cnt.subject == "ICMS92"]
t = icms.mwu(m1.loc[m1.phase == "early", "n_mod"], m1.loc[m1.phase == "late", "n_mod"], "less")
print(f"Mouse 1, 4 uA: early median {t['med1']} (n={t['n1']}), late median {t['med2']} (n={t['n2']}), "
      f"one-sided p = {t['p']:.3g}, r = {t['r']:.2f}")
res_fig4g = t
tg = icms.mwu(cnt.loc[cnt.phase == "early", "n_mod"], cnt.loc[cnt.phase == "late", "n_mod"], "less")
print(f"all six mice, 4 uA: early median {tg['med1']} (n={tg['n1']}), late median {tg['med2']} (n={tg['n2']}), "
      f"one-sided p = {tg['p']:.3g}, r = {tg['r']:.2f}")
for sub in icms.BEHAVIOUR:
    d = cnt[cnt.subject == sub]
    tt = icms.mwu(d.loc[d.phase == "early", "n_mod"], d.loc[d.phase == "late", "n_mod"], "less")
    print(f"  {icms.MOUSE[sub]}: early mean {d.loc[d.phase == 'early', 'n_mod'].mean():.2f} (n={tt['n1']}), "
          f"late mean {d.loc[d.phase == 'late', 'n_mod'].mean():.2f} (n={tt['n2']}), one-sided p = {tt['p']:.3g}")

**Result: reproduced.** In Mouse 1 at 4 uA the number of modulated responses per channel and session is higher late than early (median 12 against 14, n = 8 and 21 as in the paper, one-sided p = 0.0443, r = 0.42). Over all six mice the same test is not significant (p = 0.0728).

## fig4ij-pyramidal-modulation-increase

> Increases in modulation over training are carried by pyramidal cells, at 4, 5 and 6 uA, with
> latency decreasing at 5 and 6 uA.

z cap 100 (claims). The code's one-sided tests are a variant.

In [ ]:
M100 = R[R.mod100 & R.current.isin(icms.CURRENTS) & R.phase.notna()]
PYR = M100[M100.ctype == "pyramidal"]
res_pyr_z = phase_test(PYR, "z", "pyramidal z")
res_pyr_t = phase_test(PYR, "tmax_ms", "pyramidal time to max (ms)")
print("variant, one-sided (late greater for z, late smaller for time to max):")
_ = phase_test(PYR, "z", "pyramidal z", "less")
_ = phase_test(PYR, "tmax_ms", "pyramidal time to max", "greater")

**Result: reproduced.** Pyramidal modulation rises at 4, 5 and 6 uA (p = 4.79e-05, 0.000587, 0.000878) and time to maximum falls at 5 uA (p = 0.000329) and 6 uA (p = 0.025). It also falls at 4 uA (p = 0.0208), where the paper reports p = 0.32.

## fig4ij-interneurons-little-change

> Interneurons show no significant change in modulation at any current; their latency changes
> significantly only at 6 uA.

In [ ]:
INT = M100[M100.ctype == "interneuron"]
res_int_z = phase_test(INT, "z", "interneuron z")
res_int_t = phase_test(INT, "tmax_ms", "interneuron time to max (ms)")
print("interneuron responses by mouse:", INT.mouse.value_counts().to_dict())

**Result: reproduced.** Interneuron modulation does not change at 4, 5 or 6 uA (p = 0.331, 0.884, 0.0885; r = 0.08, -0.01, 0.15). Their time to maximum changes only at 6 uA (130 to 190 ms, p = 0.0335, r = 0.19; paper p = 0.010, r = 0.21), not at 4 or 5 uA (p = 0.929, 0.361).

## fig5d-pulse-locking-stable-across-currents

> Over 80% of units keep their PL or NPL identity at a given stimulation contact as the current
> rises from 4 to 6 uA.

In [ ]:
pat_rows = []
for (si, ch, u), g in M50.groupby(["si", "channel", "unit"]):
    if len(g) < 2:
        continue
    seq = g.sort_values("current").pl.astype(bool).tolist()
    if all(seq):
        p = "all PL"
    elif not any(seq):
        p = "all NPL"
    elif seq == sorted(seq):
        p = "NPL to PL"
    elif seq == sorted(seq, reverse=True):
        p = "PL to NPL"
    else:
        p = "mixed"
    pat_rows.append({"mouse": g.mouse.iloc[0], "pattern": p, "n_currents": len(g)})
PAT = pd.DataFrame(pat_rows)
vc = PAT.pattern.value_counts()
frac_keep = (vc.get("all PL", 0) + vc.get("all NPL", 0)) / len(PAT)
print("unit-channel pairs seen at two or more currents:", len(PAT), "| patterns:", vc.to_dict())
print(f"fraction keeping PL or NPL identity: {frac_keep:.3f}")
print("by mouse:", PAT.groupby("mouse").pattern.apply(lambda x: round(x.isin(["all PL", "all NPL"]).mean(), 3)).to_dict())

**Result: reproduced.** Of 740 unit-channel pairs seen at two or more currents (paper 729), 529 stay PL and 94 stay NPL, a fraction of 0.842, above 0.80; every mouse is between 0.814 and 0.961.

## fig5e-pyramidal-fraction-pl

> Pyramidal cells are more common among PL units (76.2%) than among NPL units (49.8%).

Two-step majority: over stimulation channels at each current, then over currents, ties NPL.
Units of unknown type are left out of the fractions.

In [ ]:
ucls = []
for (si, u), g in M50.groupby(["si", "unit"]):
    per_cur = []
    for c, gc in g.groupby("current"):
        k = int(gc.pl.astype(bool).sum())
        per_cur.append(k > len(gc) - k)
    k = sum(per_cur)
    ucls.append({"si": si, "unit": u, "ctype": g.ctype.iloc[0], "mouse": g.mouse.iloc[0],
                 "PL": k > len(per_cur) - k})
UC = pd.DataFrame(ucls)
tab5e = pd.crosstab(UC.PL, UC.ctype)
print(tab5e)
known = UC[UC.ctype != "unknown"]
f_pl = (known[known.PL].ctype == "pyramidal").mean()
f_npl = (known[~known.PL].ctype == "pyramidal").mean()
odds, p_f = stats.fisher_exact(pd.crosstab(known.PL, known.ctype).values)
print(f"pyramidal fraction: PL units {f_pl:.3f} (n={int(known.PL.sum())}), NPL units {f_npl:.3f} "
      f"(n={int((~known.PL).sum())}); Fisher exact p = {p_f:.3g}, odds ratio {odds:.2f}")
# variant: cell type from the trough-to-peak of the stored waveform
t2p_of = {(si, u): ("interneuron" if trough_to_peak_ms(sessions[si]["waveform"][u]) < 0.45 else "pyramidal")
          for si, u in zip(UC.si, UC.unit)}
UC["t2p"] = [t2p_of[(a, b)] for a, b in zip(UC.si, UC.unit)]
fp2, fn2 = (UC[UC.PL].t2p == "pyramidal").mean(), (UC[~UC.PL].t2p == "pyramidal").mean()
print(f"variant, trough-to-peak types: PL {fp2:.3f}, NPL {fn2:.3f}, Fisher p = "
      f"{stats.fisher_exact(pd.crosstab(UC.PL, UC.t2p).values)[1]:.3g}")

**Result: reproduced.** Pyramidal cells are 0.747 of PL units (n = 375) and 0.536 of NPL units (n = 110), both within tolerance, Fisher exact p = 4.93e-05. With cell types from the trough-to-peak of the stored waveform, 0.714 and 0.582 (p = 0.0103).

## figs4b-pl-latency-distance

> PL spike latency increases linearly with distance from the stimulation site, giving conduction
> velocities of about 0.2 to 0.27 m/s; latency jitter does not vary with distance.

Distance is |unit_y_um - rel_y of the stimulation contact| along the linear probe (both on the
probe axis; the stored unit_x_um is 0 and the contacts' rel_x is 0). Stimulation channels absent
from the electrode table (high-impedance contacts left out of it) cannot be placed.

In [ ]:
def stim_y(si, ch):
    s = sessions[si]
    k = np.flatnonzero(s["el_channel"] == ch)
    return s["el_y"][k[0]] if len(k) else np.nan

PLR = M50[M50.pl.astype(bool)].copy()
PLR["dist"] = [abs(sessions[si]["unit_y"][u] - stim_y(si, ch)) for si, u, ch in zip(PLR.si, PLR.unit, PLR.channel)]
print("PL responses:", len(PLR), "| without a distance (stimulation contact not in the electrode table):",
      int(PLR.dist.isna().sum()), "| distance range", PLR.dist.min().round(1), "to", PLR.dist.max().round(1), "um")
# check of the channel mapping: does modulation fall with distance from the stimulation contact?
A = R[R.current.isin(icms.CURRENTS) & R.phase.notna()].copy()
A["dist"] = [abs(sessions[si]["unit_y"][u] - stim_y(si, ch)) for si, u, ch in zip(A.si, A.unit, A.channel)]
print("all responses at 4-6 uA by distance from the stimulation contact (um):")
print(A.groupby(pd.cut(A.dist, [0, 100, 200, 400, 800, 2000])).agg(
    fraction_modulated=("mod50", "mean"), n=("mod50", "size"), median_z=("z", "median")).round(3))
edges = np.arange(60, 961, 120)
centres = (edges[:-1] + edges[1:]) / 2
res_s4b = {}
for c in icms.CURRENTS:
    d = PLR[(PLR.current == c) & PLR.dist.notna()]
    b = np.digitize(d.dist, edges) - 1
    out = {}
    for agg in ["mean", "median"]:
        y = np.array([getattr(np, agg)(d.latency_ms[b == i]) if (b == i).sum() else np.nan for i in range(len(centres))])
        use = (centres >= 120) & (centres <= 840) & ~np.isnan(y)
        lr = stats.linregress(centres[use], y[use])
        out[agg] = (lr.slope, lr.pvalue, lr.rvalue ** 2, 1e-3 / lr.slope if lr.slope > 0 else np.nan)
        print(f"{int(c)} uA latency, binned {agg}s over 120-840 um: slope {lr.slope:.5f} ms/um, p = {lr.pvalue:.3g}, "
              f"r2 = {lr.rvalue ** 2:.2f}, velocity {out[agg][3]:.3f} m/s; bins n = "
              f"{[int((b == i).sum()) for i in range(len(centres))]}")
    yj = np.array([np.mean(d.jitter_ms2[b == i]) if (b == i).sum() else np.nan for i in range(len(centres))])
    use = (centres >= 120) & (centres <= 840) & ~np.isnan(yj)
    lj = stats.linregress(centres[use], yj[use])
    lj2 = stats.linregress(d.dist, d.jitter_ms2)
    out["jitter_binned"] = (lj.slope, lj.pvalue)
    out["jitter_unbinned"] = (lj2.slope, lj2.pvalue)
    print(f"   jitter: binned means slope {lj.slope:.2e} ms2/um p = {lj.pvalue:.3g}; unbinned slope {lj2.slope:.2e} "
          f"p = {lj2.pvalue:.3g}; latency range {d.latency_ms.min():.2f} to {d.latency_ms.max():.2f} ms")
    res_s4b[c] = out

**Result: not reproduced.** PL latency does not increase with distance from the stimulation contact: the slope over 120 to 840 um is negative at 4 uA (-0.00108 ms/um on binned means, p = 0.0202) and not significant at 5 and 6 uA (p = 0.955 and 0.708), so no conduction velocity can be formed. Jitter shows no slope (p 0.51 to 0.985), as the paper reports. The fraction of modulated responses does not fall with distance either (0.406 within 100 um, 0.524 at 400 to 800 um), so the outputs cannot show that the stored stimulation channel numbers and the electrode positions are matched as this analysis assumes.

## fig5fg-pl-modulation-increase

> PL modulation over the 700 ms train increases from early to late at 4, 5 and 6 uA, and PL time to
> maximum firing rate decreases at 5 and 6 uA.

In [ ]:
PLm = M50[M50.pl.astype(bool)]
NPLm = M50[~M50.pl.astype(bool)]
res_pl_z = phase_test(PLm, "z", "PL z")
res_pl_t = phase_test(PLm, "tmax_ms", "PL time to max (ms)")

**Result: partially reproduced.** PL modulation rises at 4, 5 and 6 uA (p = 0.000361, 0.00168, 2.61e-05). PL time to maximum falls at 5 uA (75 to 35 ms, p = 0.00455) but not significantly at 6 uA (95 to 55 ms, p = 0.0798). The PL counts (159/587, 226/402, 238/194) are above the paper's (136/469, 188/285, 179/135): more responses are called PL here.

## figs4de-npl-no-change

> NPL responses show no significant change in modulation or time to maximum firing rate over
> training at any current.

In [ ]:
res_npl_z = phase_test(NPLm, "z", "NPL z")
res_npl_t = phase_test(NPLm, "tmax_ms", "NPL time to max (ms)")

**Result: not reproduced.** NPL modulation rises at 4 uA (2.526 to 3.340, n = 32 and 151, p = 0.0416, r = 0.23); at 5 and 6 uA it does not change (p = 0.822, 0.847), and time to maximum does not change at any current (p = 0.164, 0.991, 0.887).

## fig5h-pl-spike-probability-increase

> PL responses show increased peak spike probability within the 10 ms interpulse interval over
> training at 5 and 6 uA, not at 4 uA.

In [ ]:
res_pl_pk = phase_test(PLm, "peak_prob", "PL peak spike probability")
print("peak probability range:", PLm.peak_prob.min().round(4), "to", PLm.peak_prob.max().round(4))

**Result: partially reproduced.** The PL peak spike probability rises at 6 uA (0.036 to 0.044, p = 0.0359, r = 0.12) but not at 5 uA (p = 0.861); at 4 uA it does not change (p = 0.905), as the paper reports.

## figs5-restricted-window-consistent

> In a 120 ms window after stimulation onset, before movement, PL modulation still increases at
> all currents while NPL modulation does not change, and PL time to maximum falls at 4 and 5 uA.

Spikes in the first 120 ms of the train against the 120 ms before it; time to maximum in 2 ms
bins smoothed by a 5 ms Gaussian. Readings: (keep) modulated set and PL labels from the 700 ms
window; (recompute) modulated by the 120 ms test and PL from the first 12 pulses.

In [ ]:
t0 = time.time()
r120 = icms.response_table(sessions, rng_seed=3, currents=icms.CURRENTS, window=0.12, tmax_bin=0.002, tmax_sigma=0.005)
key700 = {(r["si"], r["unit"], r["channel"], r["current"]): r for r in resp_rows}
rng120 = np.random.default_rng(4)
for r in r120:
    r["mod120"] = icms.modulated(r, 50, True, ch_days)
    k = key700[(r["si"], r["unit"], r["channel"], r["current"])]
    r["mod700"], r["pl700"] = k["mod50"], bool(k.get("pl", False))
    if r["mod120"]:
        s = sessions[r["si"]]
        tr = icms.ephys_trains(s)
        m = (tr["stim_channel"] == r["channel"]) & (tr["current_uA"] == r["current"])
        Mx = icms.pulse_matrix(s["spikes"][r["unit"]], np.sort(tr["start_time"][m]), n_pulses=12)
        r["pl120"] = icms.pulse_locking(Mx, rng120)["pl"]
R120 = pd.DataFrame(r120)
R120 = R120[R120.phase.notna()]
print(f"120 ms responses: {len(R120)} in {time.time() - t0:.0f} s; modulated in 120 ms: {int(R120.mod120.sum())}")
res_s5 = {}
for reading, sel, lab in [("keep", R120.mod700, "pl700"), ("recompute", R120.mod120, "pl120")]:
    d = R120[sel]
    pl = d[d[lab].astype(bool)]
    npl = d[~d[lab].astype(bool)]
    print(f"\nreading {reading}:")
    res_s5[reading] = {"pl_z": phase_test(pl, "z", "PL z"), "npl_z": phase_test(npl, "z", "NPL z"),
                       "pl_t": phase_test(pl, "tmax_ms", "PL time to max"),
                       "npl_t": phase_test(npl, "tmax_ms", "NPL time to max")}

**Result: reproduced, under the reading that keeps the 700 ms labels.** In the first 120 ms, PL modulation rises at 4, 5 and 6 uA (p = 0.00222, 0.000301, 0.000849), NPL modulation does not change (p = 0.0689, 0.563, 0.592), and PL time to maximum falls at 4 and 5 uA (p = 0.0244, 0.0183) and not at 6 uA (p = 0.908), as the paper reports. With the modulated set and labels recomputed in the 120 ms window, NPL modulation rises at all three currents (p = 0.0132, 0.00291, 0.0187).

## fig6c-kl-divergence

> Hit-miss separation of delta-spikes distributions (KL divergence) increases over time for both
> PL and NPL responses, and NPL responses separate hit from miss more than PL responses in both
> early and late phases.

Trials of the modulated responses (mod50) at 4 to 6 uA, pooled per week and label. Readings (k1)
winsorizing 1%, fraction 0.4 and (k2) 2.5%, fraction 0.8; 1000 draws.

In [ ]:
lab_of = {(r.si, r.unit, r.channel, r.current): bool(r.pl) for r in M50.itertuples()}
trial_pool = defaultdict(lambda: {"hit": [], "miss": []})
for r in resp_rows:
    k = (r["si"], r["unit"], r["channel"], r["current"])
    if k not in lab_of or r["phase"] is None:
        continue
    key = (r["week"], "PL" if lab_of[k] else "NPL")
    trial_pool[key]["hit"].extend(r["trial_delta"][r["trial_hit"]].tolist())
    trial_pool[key]["miss"].extend(r["trial_delta"][~r["trial_hit"]].tolist())
print("trials per week and label (hit, miss):", {k: (len(v["hit"]), len(v["miss"])) for k, v in sorted(trial_pool.items())})
res_kl = {}
for reading, wins, frac in [("k1", 0.01, 0.4), ("k2", 0.025, 0.8)]:
    rngk = np.random.default_rng(10)
    kl = {k: icms.kl_bootstrap(v["hit"], v["miss"], rngk, wins, frac) for k, v in sorted(trial_pool.items())}
    def pool(phase, lab):
        return np.concatenate([kl[(w, lab)] for w in (icms.EARLY_WEEKS if phase == "early" else icms.LATE_WEEKS)
                               if (w, lab) in kl])
    tests = {"PL early vs late": icms.mwu(pool("early", "PL"), pool("late", "PL")),
             "NPL early vs late": icms.mwu(pool("early", "NPL"), pool("late", "NPL")),
             "NPL vs PL early": icms.mwu(pool("early", "PL"), pool("early", "NPL")),
             "NPL vs PL late": icms.mwu(pool("late", "PL"), pool("late", "NPL"))}
    q = icms.fdr_bh([t["p"] for t in tests.values()])
    print(f"\nreading {reading}: median KL per week and label:",
          {f"w{w} {l}": round(float(np.median(v)), 4) for (w, l), v in kl.items()})
    for (name, t), qq in zip(tests.items(), q):
        print(f"  {name}: medians {t['med1']:.4f} vs {t['med2']:.4f} (n={t['n1']}, {t['n2']}), FDR p = {qq:.3g}, r = {t['r']:.2f}")
    res_kl[reading] = (tests, q)

**Result: not reproduced.** NPL responses separate hit from miss more than PL in both phases (reading k1, median KL 0.3616 against 0.0400 early and 0.3321 against 0.0356 late, FDR p = 0), but the separation does not rise over training: PL falls from 0.0400 to 0.0356 (FDR p = 1.09e-43, r = -0.23) and NPL from 0.3616 to 0.3321 (FDR p = 0.0218). Reading k2 gives the same directions (PL 0.0503 to 0.0354; NPL 0.2582 to 0.2437, FDR p = 0.176).

## fig6d-npl-hit-miss-divergence

> PL responses show no significant hit-miss divergence in firing rate at any time, while NPL
> responses diverge from 65 ms after ICMS onset early in training and from 35 ms late in
> training, both before movement onset.

Each row is one modulated response's trial-averaged rate for hit and for miss trials (responses
with at least one of each), from train onset to 700 ms. Labels are permuted by swapping a
response's hit and miss rows, which permutes the labels over rows while keeping each response's
pair (the code's permutation of hit and miss labels over rows; the exact scheme is not in the
paper). 1000 permutations. Readings (b1) 10 ms bins and (b2) 5 ms bins, onset at the bin centre;
(o1) first and (o2) last significant cluster.

In [ ]:
t0 = time.time()
res_6d = {}
for bin_s, bl in [(0.010, "b1"), (0.005, "b2")]:
    rows_h = defaultdict(list)
    rows_m = defaultdict(list)
    for r in resp_rows:
        k = (r["si"], r["unit"], r["channel"], r["current"])
        if k not in lab_of or r["phase"] is None or r["trial_hit"].sum() == 0 or (~r["trial_hit"]).sum() == 0:
            continue
        spk = sessions[r["si"]]["spikes"][r["unit"]]
        key = (r["phase"], "PL" if lab_of[k] else "NPL")
        rows_h[key].append(icms.psth(spk, r["starts"][r["trial_hit"]], bin_s))
        rows_m[key].append(icms.psth(spk, r["starts"][~r["trial_hit"]], bin_s))
    for key in sorted(rows_h):
        rngc = np.random.default_rng(20)
        H, Mm = np.array(rows_h[key]), np.array(rows_m[key])
        cl, tstat = icms.cluster_test(H, Mm, rngc, n_perm=1000)
        sig = [c for c in cl if c["p"] < 0.05]
        on1 = (sig[0]["start_bin"] + 0.5) * bin_s * 1000 if sig else np.nan
        on2 = (sig[-1]["start_bin"] + 0.5) * bin_s * 1000 if sig else np.nan
        res_6d[(bl, key)] = (len(H), sig, on1, on2)
        print(f"{bl} {key[0]} {key[1]}: rows {len(H)}, clusters {len(cl)}, significant "
              f"{[(round((c['start_bin'] + 0.5) * bin_s * 1000, 1), round((c['end_bin'] + 0.5) * bin_s * 1000, 1), round(c['mass'], 1), c['p']) for c in sig]}; "
              f"onset first {on1} ms, last {on2} ms")
print(f"cluster tests in {time.time() - t0:.0f} s")

**Result: not reproduced.** PL responses diverge between hit and miss in both phases (significant clusters from 95 ms early and from 5 ms late with 10 ms bins). NPL onsets are 105 ms early and 55 ms late with 10 ms bins (first cluster), and 102.5 and 27.5 ms with 5 ms bins; the early onset is outside 65 +- 10 ms under every reading, though late is earlier than early under all of them.

## fig6f-population-coupling

> PL population coupling increases from early to late while NPL coupling stays stable, and NPL
> units have higher population coupling than PL units in both phases.

In [ ]:
def coupling_tests(rows_c, label):
    C = pd.DataFrame(rows_c)
    C = C[C.phase.notna() & C.coupling.notna()]
    C["lab"] = np.where(C.pl.astype(bool), "PL", "NPL")
    tests = {"PL early vs late": icms.mwu(C[(C.lab == "PL") & (C.phase == "early")].coupling,
                                          C[(C.lab == "PL") & (C.phase == "late")].coupling),
             "NPL early vs late": icms.mwu(C[(C.lab == "NPL") & (C.phase == "early")].coupling,
                                           C[(C.lab == "NPL") & (C.phase == "late")].coupling),
             "NPL vs PL early": icms.mwu(C[(C.lab == "PL") & (C.phase == "early")].coupling,
                                         C[(C.lab == "NPL") & (C.phase == "early")].coupling),
             "NPL vs PL late": icms.mwu(C[(C.lab == "PL") & (C.phase == "late")].coupling,
                                        C[(C.lab == "NPL") & (C.phase == "late")].coupling)}
    q = icms.fdr_bh([t["p"] for t in tests.values()])
    print(f"{label}: values {len(C)}, range {C.coupling.min():.2f} to {C.coupling.max():.2f}")
    for (name, t), qq in zip(tests.items(), q):
        print(f"  {name}: medians {t['med1']:.3f} vs {t['med2']:.3f} (n={t['n1']}, {t['n2']}), FDR p = {qq:.3g}, r = {t['r']:.2f}")
    print("  median by mouse and label:", C.groupby([C.subject.map(icms.MOUSE), "lab"]).coupling.median().round(2).to_dict())
    return tests, q

for r in resp_rows:
    r["mod"] = r["mod50"]  # population and units: the mod50 responses
t0 = time.time()
pc_main = icms.population_coupling(sessions, resp_rows, rng_seed=30)
res_6f = coupling_tests(pc_main, "700 ms, all six mice")
print(f"in {time.time() - t0:.0f} s")

**Result: reproduced.** PL coupling rises from early to late (median 0.271 to 0.688, FDR p = 0.00158, r = 0.09), NPL coupling does not change (1.681 to 1.509, FDR p = 0.49), and NPL exceeds PL early (FDR p = 3.39e-12, r = 0.37) and late (FDR p = 2.92e-12, r = 0.26). The PL n (621, 1178) exceed the paper's (494, 899) and the NPL n (152, 309) are below them (263, 553).

## figs9a-coupling-700ms

> In the 700 ms window of fig. S9A, PL population coupling increases from early to late while NPL
> coupling does not change, and PL coupling is lower than NPL coupling in both phases.

Readings (a) as fig6f without Mouse 6, (b) the population built from the other modulated PL
units only.

In [ ]:
pc_a = [r for r in pc_main if r["subject"] != "ICMS83"]
res_s9a_a = coupling_tests(pc_a, "(a) without Mouse 6")
pc_b = icms.population_coupling(sessions, resp_rows, rng_seed=31, pop_label=True)
res_s9a_b = coupling_tests(pc_b, "(b) population of PL units only")

**Result: reproduced.** Without Mouse 6 (reading a), PL coupling rises (0.180 to 0.728, FDR p = 1.95e-05), NPL does not change (FDR p = 0.475), and PL is below NPL early (FDR p = 1.26e-12) and late (FDR p = 9.07e-11). The PL n (567, 1038) exceed the legend's 446 and 763. Reading (b) gives the same four results (FDR p = 0.00105, 0.193, 3.48e-13, 3.54e-11).

## figs9-coupling-window-and-control (120 ms part)

The control part of this claim is blocked (see the summary); the 120 ms part is run here and
reported. Spikes in the first 120 ms of each train, a 10-spike minimum, labels from the 700 ms
window.

In [ ]:
pc_120 = icms.population_coupling(sessions, resp_rows, rng_seed=32, window=0.12, min_spikes=10)
res_s9_120 = coupling_tests(pc_120, "120 ms window")

**Result: blocked for the control part; the 120 ms part holds.** In the first 120 ms, NPL coupling exceeds PL early (0.752 against -0.323, FDR p = 2.04e-07, r = 0.28) and late (0.653 against -0.233, FDR p = 1.27e-12, r = 0.28). The control test is not run (see the summary).

## Control animals (data check for the blocked control claims)

The control files are read here only to describe them: their stimulation protocol, their imaging
tables and what the control claims would need. No outcome is decided from this cell except
through the blocks stated in the summary.

In [ ]:
ctrl = [a for a in assets if a["path"].split("/")[0][4:] not in icms.BEHAVIOUR]
crow = []
for a in ctrl:
    f = icms.open_asset(a["asset_id"])
    d = {"file": a["path"].split("/")[1], "date": f["session_start_time"][()].decode()[:10],
         "subject": f["general/subject/subject_id"][()].decode()}
    if "intervals/electrical_stimulation" in f:
        st = f["intervals/electrical_stimulation"]
        d["pulses_per_burst"] = ",".join(map(str, np.unique(st["pulse_count"][()])))
        d["freq_hz"] = ",".join(map(str, np.unique(st["frequency_hz"][()])))
        d["bursts_per_trial"] = int(np.max(st["burst_index_in_trial"][()]) + 1) if "burst_index_in_trial" in st else None
        d["burst_ms"] = float(np.round(np.median(st["stop_time"][()] - st["start_time"][()]) * 1000, 1))
        d["units"] = len(f["units/id"]) if "units" in f else 0
        d["cell_types"] = ",".join(np.unique(f["units/cell_type"][()].astype(str))) if "units" in f else ""
    o = "processing/ophys"
    if o + "/stimulus_conditions" in f:
        g = f[o + "/ImageSegmentation_ControlSurvey/PlaneSegmentation_ControlSurvey"]
        act = g["activation"][()]
        cur = f[o + "/stimulus_conditions/current_uA"][()]
        d["survey_rois"] = act.shape[0]
        d["currents_imaged"] = ",".join(str(int(c)) for c in np.unique(cur))
        d["activated_at_5uA_per_channel"] = ",".join(str(int(v)) for v in act[:, cur == 5].sum(0))
        ip = f["general/optophysiology/ImagingPlane_ControlSurvey"]
        d["grid_spacing_given"] = "grid_spacing" in ip
        d["stim_contacts_at_origin"] = int(((f[o + "/stim_electrode_positions/x"][()] == 0) &
                                            (f[o + "/stim_electrode_positions/y"][()] == 0)).sum())
    f.close()
    crow.append(d)
CT = pd.DataFrame(crow).sort_values(["subject", "date"])
print(CT.to_string())
print("\ncontrol mice:", CT.subject.nunique(), "| control files:", len(CT), "| with survey imaging:",
      int(CT.survey_rois.notna().sum()), "| with units:", int((CT.get("units", pd.Series(dtype=float)).fillna(0) > 0).sum()))

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig2a-threshold-decrease | reproduced |
| fig2a-steepest-decline-early | reproduced (rests on the criterion's reading of the figure) |
| figs1e-response-times-decrease | reproduced |
| figs5c-wheel-onset-decrease | reproduced |
| fig4-unit-counts | partially reproduced |
| figs3b-modulation-increase-all | reproduced |
| figs3c-latency-all | partially reproduced |
| figs3d-modulation-stable-at-threshold | reproduced |
| fig4g-modulated-count-mouse1 | reproduced |
| fig4ij-pyramidal-modulation-increase | reproduced |
| fig4ij-interneurons-little-change | reproduced |
| fig5d-pulse-locking-stable-across-currents | reproduced |
| fig5e-pyramidal-fraction-pl | reproduced |
| figs4b-pl-latency-distance | not reproduced |
| fig5fg-pl-modulation-increase | partially reproduced |
| figs4de-npl-no-change | not reproduced |
| fig5h-pl-spike-probability-increase | partially reproduced |
| figs5-restricted-window-consistent | reproduced |
| fig6c-kl-divergence | not reproduced |
| fig6d-npl-hit-miss-divergence | not reproduced |
| fig6f-population-coupling | reproduced |
| figs9a-coupling-700ms | reproduced |
| figs9-coupling-window-and-control | blocked (methods underspecified for the controls) |
| fig2c, fig2b, figs2ab, fig2e (two), figs2de-behaviour, fig2g | blocked (metadata insufficient) |
| figs2de-control-stable | blocked (metadata insufficient) |
| fig3b, fig3df, fig3i, fig3j (two), fig3lm, fig3n (two), figs6 | blocked (metadata insufficient) |
| fig4d-waveform-separation, figs8-tracked-units | blocked (data missing) |
| figs7-control-no-change | blocked (methods underspecified) |

How the analysis differs from the paper's. Thresholds are refitted with the Python port of
psignifit on a coarser grid; the dataset stores none. The movement-onset threshold is taken from
hit trials, as miss trials have no response time. Pulse-locking follows the claims file's reading
of the code; it calls more responses PL than the paper (for example 159 against 136 early PL
responses at 4 uA). The hit-miss PSTH bin width and smoothing are not in the paper or its code;
two widths were run. Unit distances to the stimulation contact assume that the stored
stimulation channel numbers index the electrode table's `channel_name`.

What was found about the dataset. The 46 sessions of weeks 0 to 4 match the paper's count,
and the ephys response counts are close to the paper's, but the unit total (795 in weeks 0 to 4)
is below the paper's 973. Behavioural files give no stimulation-site position in the imaging
volume and no activation calls, and the planar ROIs are segmented per session with no identity
across sessions, so no imaging claim can be tested. The control files use 3 bursts of 10 pulses
at 50 Hz, where the paper says the controls had similar stimulation parameters. Most miss trials carry
no response time, Mouse 6 has no catch trials, and cell types are spelled differently in Mouse 6.